In [4]:
import  sklearn
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd
import warnings
from sklearn import preprocessing
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
import shap
import copy
import matplotlib.pyplot as plt
from lightgbm import LGBMRegressor
from sklearn import datasets
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import time
import lightgbm as lgb
from sklearn.model_selection import KFold
import torch
import torch.optim as optim
from sklearn.ensemble import RandomForestRegressor
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import copy
import numpy as np
from xgboost import DMatrix, train
from sklearn.datasets import make_regression
import torch.optim as optim
np.random.seed(42)
torch.manual_seed(42)
warnings.filterwarnings("ignore")
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import matplotlib
import pickle
import glob
import numpy as np
from scipy.stats import *
import xgboost as xgb
import torch.nn.functional as F
import numpy as np
from scipy.stats import mode
import copy
from torch.utils.data import Dataset, DataLoader
import torch
# from pytorch_tabnet.tab_model import TabNetRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_squared_error
import pickle
matplotlib.rcParams['font.sans-serif'] = ['SimHei']  
matplotlib.rcParams['axes.unicode_minus'] = False 
import os
import gc
import numpy as np
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from flaml import AutoML

import warnings
warnings.filterwarnings("ignore", category=UserWarning)
import pickle
def set_seed(seed):
    import random
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(42)
from sklearn.utils import shuffle
import pickle
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# Data generation

# Common methods

In [ ]:
"""
Slope and Curvature 
"""
def compute_slope_curvature_rmse(Numpy_HRDEM, Numpy_CorDEM, cellsize_x=30, cellsize_y=30):

    def calculate_slope_curvature(dem, cellsize_x, cellsize_y):
    
        a = dem[:, :-2, :-2]  
        b = dem[:, :-2, 1:-1] 
        c = dem[:, :-2, 2:] 
        d = dem[:, 1:-1, :-2] 
        e = dem[:, 1:-1, 1:-1] 
        f = dem[:, 1:-1, 2:]  
        g = dem[:, 2:, :-2]
        h = dem[:, 2:, 1:-1]
        i = dem[:, 2:, 2:] 

        #  dz/dx
        dz_dx = ((c + 2 * f + i) - (a + 2 * d + g)) / (8 * cellsize_x)
        #  dz/dy
        dz_dy = ((g + 2 * h + i) - (a + 2 * b + c)) / (8 * cellsize_y)

      
        rise_run = np.sqrt(dz_dx ** 2 + dz_dy ** 2)
        slope = np.arctan(rise_run) * 57.29578  

    
        curvature = dz_dx ** 2 + dz_dy ** 2  

        return slope, curvature

 
    slope_ref, curvature_ref = calculate_slope_curvature(Numpy_HRDEM, cellsize_x, cellsize_y)
  
    slope_est, curvature_est = calculate_slope_curvature(Numpy_CorDEM, cellsize_x, cellsize_y)


    slope_rmse = np.sqrt(np.mean((slope_est - slope_ref) ** 2))
    curvature_rmse = np.sqrt(np.mean((curvature_est - curvature_ref) ** 2))

    return slope_rmse, curvature_rmse



# Predictions

In [6]:
import numpy as np
from osgeo import gdal

def AssignValuesByPixelID(input_raster_path, output_raster_path, pid_array, value_array):
    dataset = gdal.Open(input_raster_path, gdal.GA_ReadOnly)
    band = dataset.GetRasterBand(1)
    pixel_array = band.ReadAsArray()
    nodata = band.GetNoDataValue()


 
    if nodata is not None:
        pixel_array = np.where(pixel_array == nodata, 0, pixel_array)  

    pixel_id_array = pixel_array.astype(np.int32)

    
    lut_size = pixel_id_array.max() + 1
    
    lut = np.full(lut_size, nodata, dtype=np.float32)

   
    pid_array = pid_array.astype(np.int32)

    lut[pid_array] = value_array

 
    mapped_array = lut[pixel_id_array]


    driver = gdal.GetDriverByName('GTiff')
    outDataset = driver.Create(output_raster_path,
                               pixel_id_array.shape[1], pixel_id_array.shape[0],
                               1, gdal.GDT_Float32)
    outDataset.SetGeoTransform(dataset.GetGeoTransform())
    outDataset.SetProjection(dataset.GetProjection())
    outBand = outDataset.GetRasterBand(1)
    outBand.WriteArray(mapped_array)
    outBand.SetNoDataValue(nodata)
    outBand.FlushCache()

    dataset, band, outDataset, outBand = None, None, None, None
    return output_raster_path


In [7]:
test_labels_1d,test_ATL_1d,test_DEM_1d,test_FABDEM_1d,test_Fathom_1d,test_GEDTM30_1d=None,None,None,None,None,None
raw_rmse,raw_slopeRMSE,raw_curvatureRMSE=None,None,None
train_data_2d,val_data_2d,test_data_2d,train_labels_1d,val_labels_1d=None,None,None,None,None
from sklearn.utils import shuffle


npzFolder=r"DataModels"  

label_size=64

label_area="NewZeeLand_nowater"

testPath_1=os.path.join(npzFolder,f"{label_area}_test_HM_{label_size}.npz")
otherPath_1=os.path.join(npzFolder,f"{label_area}_other_HM_{label_size}.npz")

label_area="America_nowater"

testPath_2=os.path.join(npzFolder,f"{label_area}_test_HM_{label_size}.npz")
otherPath_2=os.path.join(npzFolder,f"{label_area}_other_HM_{label_size}.npz")

label_area="European_nowater"

testPath_3=os.path.join(npzFolder,f"{label_area}_test_HM_{label_size}.npz")
otherPath_3=os.path.join(npzFolder,f"{label_area}_other_HM_{label_size}.npz")


usingFeatures = [
    "IDW", "Lon", "Lat","LandCover", 
      "ForestCoverage", "ForestHeight", "ForestHeight10", "ForestHeight30",  "BuildingHeight90", "Population", "NightLight",  "Aspect",
      "Hillshade", "Curvature", "ProfileCurvature", "PlanCurvature",  "DEM", "STDDEM", "Max_MinDEM", "Mean_MinDEM", "deltMeanDEM", "deltMedianDEM", "deltMinDEM", "deltMaxDEM",
    "Slope", "MaxSlope", "STDSlope", "Mean_MinSlope", "Max_MinSlope", "deltMeanSlope", "SOS", "SOA","HeightError","FCLoss","FCGain","NIR","RGB","SAR_VV","SAR_VH"
]

selected_features=['BuildingHeight90', 'ForestHeight', 'ForestHeight10', 'ForestHeight30', 'DEM', 'HeightError', 
                   'ForestCoverage','Aspect', 'Hillshade', 'NIR','deltMinDEM' ,'RGB', 'SAR_VV','Curvature','FCLoss']




def getTrainingData_2d(paths, mean_std_path, IsShuffe=None):
    
    
    selected_indices = [usingFeatures.index(feat) for feat in selected_features]
    
    mean_val_all=np.load(mean_std_path)['mean']
    std_val_all=np.load(mean_std_path)['std']

    mean_val = mean_val_all[0, selected_indices, 0, 0]  # shape: (num_selected_features,)
    std_val = std_val_all[0, selected_indices, 0, 0]    # shape: (num_selected_features,)

    all_data = []
    all_labels = []

    for path in paths:
        loaded = np.load(path)
        data = loaded['arr1'][:, selected_indices, :, :]  
        labels = loaded['arr2']

        for i in range(data.shape[1]):  # data.shape[1] is the number of features (channels)
            data[:, i, :, :] = (data[:, i, :, :] - mean_val[i]) / std_val[i]

        all_data.append(data)
        all_labels.append(labels)

    data = np.concatenate(all_data, axis=0)
    label = np.concatenate(all_labels, axis=0)

    B, C, H, W = data.shape
    data_reshaped = data.transpose(0, 2, 3, 1).reshape(-1, C)  # (B*H*W, C)
    label_reshaped = label.reshape(-1)  # (B*H*W,)

    if IsShuffe:
        data_reshaped, label_reshaped = shuffle(data_reshaped, label_reshaped, random_state=42)

    return data_reshaped,label_reshaped


    

def load_other_data(otherPaths):
    
    global test_DEM,test_HRDEM
    global test_DEM_1d,test_HRDEM_1d
    global RMSE_RawDEM,slope_rmse_rawDEM, curvature_rmse_rawDEM

    test_HRDEM_list = []
    test_DEM_list = []
    test_test_FABDEM_list=[]
    test_FathomDEM_list=[]
    test_GEDTM30_list=[]
    for path in otherPaths:
        loaded = np.load(path)
        test_HRDEM_list.append(loaded['hrdem'])
        test_DEM_list.append(loaded['dem'])
        test_test_FABDEM_list.append(loaded['fab'])
        test_FathomDEM_list.append(loaded['fathom'])
        test_GEDTM30_list.append(loaded['gedtm30'])


    test_HRDEM = np.concatenate(test_HRDEM_list, axis=0)
    test_DEM = np.concatenate(test_DEM_list, axis=0)
    test_FABDEM = np.concatenate(test_test_FABDEM_list, axis=0)
    test_FathomDEM = np.concatenate(test_FathomDEM_list, axis=0)
    test_GEDTM30 = np.concatenate(test_GEDTM30_list, axis=0)

    test_DEM_1d=test_DEM.reshape(-1)
    test_HRDEM_1d=test_HRDEM.reshape(-1)
    test_FABDEM_1d=test_FABDEM.reshape(-1)
    test_FathomDEM_1d=test_FathomDEM.reshape(-1)
    test_GEDTM30_1d=test_GEDTM30.reshape(-1)
 

    RMSE_RawDEM = mean_squared_error(test_HRDEM_1d, test_DEM_1d) ** 0.5
    slope_rmse_rawDEM, curvature_rmse_rawDEM = compute_slope_curvature_rmse(test_HRDEM, test_DEM)
    print(f"---RawDEM RMSE  Elevation: {RMSE_RawDEM:0.2f},  Slope: {slope_rmse_rawDEM:0.2f}, Curvature: {curvature_rmse_rawDEM:0.4f}")
    
    
    ## FABDEM
    RMSE_FAB = mean_squared_error(test_HRDEM_1d, test_FABDEM_1d) ** 0.5
    slope_rmse_FAB, curvature_rmse_FAB = compute_slope_curvature_rmse(test_HRDEM, test_FABDEM)
    print(f"---FABDEM RMSE  Elevation: {RMSE_FAB:0.2f},  Slope: {slope_rmse_FAB:0.2f}, Curvature: {curvature_rmse_FAB:0.4f}")
    
    ele_imp_FAB=((RMSE_RawDEM - RMSE_FAB) / RMSE_RawDEM) * 100
    slope_imp_FAB=100*(slope_rmse_rawDEM - slope_rmse_FAB) / slope_rmse_rawDEM
    cur_imp_FAB=100*(curvature_rmse_rawDEM - curvature_rmse_FAB) / curvature_rmse_rawDEM
    print(f'---FABDEM  improvement--   Elevation:{ele_imp_FAB:0.2f}%    Slope:{slope_imp_FAB:0.2f}%   Curvature:{cur_imp_FAB:0.2f}%')

    ## GEDTM30
    RMSE_GEDTM = mean_squared_error(test_HRDEM_1d, test_GEDTM30_1d*10) ** 0.5
    slope_rmse_GEDTM, curvature_rmse_GEDTM = compute_slope_curvature_rmse(test_HRDEM, test_GEDTM30*10)
    print(f"---GEDTM30 RMSE  Elevation: {RMSE_GEDTM:0.2f},  Slope: {slope_rmse_GEDTM:0.2f}, Curvature: {curvature_rmse_GEDTM:0.4f}")
    
    ele_imp_GEDTM=((RMSE_RawDEM - RMSE_GEDTM) / RMSE_RawDEM) * 100
    slope_imp_GEDTM=100*(slope_rmse_rawDEM - slope_rmse_GEDTM) / slope_rmse_rawDEM
    cur_imp_GEDTM=100*(curvature_rmse_rawDEM - curvature_rmse_GEDTM) / curvature_rmse_rawDEM
    print(f'---GEDTM30DEM  improvement--   Elevation:{ele_imp_GEDTM:0.2f}%    Slope:{slope_imp_GEDTM:0.2f}%   Curvature:{cur_imp_GEDTM:0.2f}%')
    
    ## Fathom
    RMSE_Fath = mean_squared_error(test_HRDEM_1d, test_FathomDEM_1d) ** 0.5
    slope_rmse_Fath, curvature_rmse_Fath = compute_slope_curvature_rmse(test_HRDEM, test_FathomDEM)
    print(f"---Fathom RMSE  Elevation: {RMSE_Fath:0.2f},  Slope: {slope_rmse_Fath:0.2f}, Curvature: {curvature_rmse_Fath:0.4f}")

    ele_imp_Fath=((RMSE_RawDEM - RMSE_Fath) / RMSE_RawDEM) * 100
    slope_imp_Fath=100*(slope_rmse_rawDEM - slope_rmse_Fath) / slope_rmse_rawDEM
    cur_imp_Fath=100*(curvature_rmse_rawDEM - curvature_rmse_Fath) / curvature_rmse_rawDEM
    print(f'---FathomDEM  improvement--   Elevation:{ele_imp_Fath:0.2f}%    Slope:{slope_imp_Fath:0.2f}%   Curvature:{cur_imp_Fath:0.2f}%')

    return test_DEM_1d-test_FathomDEM_1d, test_DEM_1d-test_FABDEM_1d,  test_DEM_1d-test_GEDTM30_1d




# Models

In [8]:
def getImp_2d(prediction_test_1d,test_labels_1d):
    predicted_rmse = np.sqrt(mean_squared_error(test_labels_1d, prediction_test_1d))
    B,H,W=test_HRDEM.shape
    pred_BHW = prediction_test_1d.reshape(B, H, W)
    slope_rmse, curvature_rmse=compute_slope_curvature_rmse(test_HRDEM, test_DEM-pred_BHW, cellsize_x=30, cellsize_y=30)
    print(f"-----Raw RMSE--- Elevation:{predicted_rmse:0.2f}   Slope:{slope_rmse:0.2f}   Curvature:{curvature_rmse:0.4f} ")
    
    print(f"-----RMSE Improvement--- Elevation:{100*(RMSE_RawDEM-predicted_rmse)/RMSE_RawDEM:0.2f}%   \
          Slope:{100*(slope_rmse_rawDEM-slope_rmse)/slope_rmse_rawDEM:0.2f}%  \
          Curvature:{100*(curvature_rmse_rawDEM-curvature_rmse)/curvature_rmse_rawDEM:0.2f}% ")




## ML models

In [9]:
import pickle

def pred_val_MLModel(model=None, modelPath=None):
   
    with open(modelPath, "rb") as f:
        automl = pickle.load(f)

    print(f"---------Model: {model}---------")

    
    y_pred = automl.predict(test_data_2d)


    predicted_rmse = np.sqrt(mean_squared_error(test_labels_1d, y_pred))
    print(f"-----Raw RMSE--- Elevation:{predicted_rmse:0.2f}  ")

    RMSE_RawDEM = np.sqrt(mean_squared_error(test_labels_1d, np.zeros_like(test_labels_1d)))
    print(f"-----RMSE Improvement--- Elevation:{100*(RMSE_RawDEM-predicted_rmse)/RMSE_RawDEM:0.2f}%  ")

    getImp_2d(prediction_test_1d=y_pred,test_labels_1d=test_labels_1d)

    return y_pred


import pickle
device = 'cuda' if torch.cuda.is_available() else 'cpu'

def predANN(modelPath=None):

    print("ANN...........")
    
    with open(modelPath, 'rb') as f:
        best_model = pickle.load(f)
            

    batch_size = 10000
    results = []
    for i in range(0, len(test_data_2d), batch_size):
        batch = test_data_2d[i:i+batch_size]
        batch_pred = best_model.predict(batch)
        results.append(batch_pred)
    prediction_test_1d = np.concatenate(results)

    predicted_rmse = np.sqrt(mean_squared_error(test_labels_1d, prediction_test_1d))
    print(f"-----Raw RMSE--- Elevation:{predicted_rmse:0.2f}  ")

    getImp_2d(prediction_test_1d,test_labels_1d)

    return prediction_test_1d





In [ ]:
# Test data is available
 
testPaths =  [testPath_1] 
otherPaths= [otherPath_1]

mean_std_path=r"DataModels/mean_std.npz"
test_data_2d,test_labels_1d=getTrainingData_2d(testPaths, mean_std_path) # Ignore GEDTM30  its previous data

pred_Fathom,pred_FABDEM,pred_GEDTM30=load_other_data(otherPaths)



---RawDEM RMSE  Elevation: 11.04,  Slope: 2.58, Curvature: 0.0650
---FABDEM RMSE  Elevation: 5.52,  Slope: 2.69, Curvature: 0.0682
---FABDEM  improvement--   Elevation:50.00%    Slope:-4.09%   Curvature:-4.97%
---GEDTM30 RMSE  Elevation: 6.07,  Slope: 3.03, Curvature: 0.0754
---GEDTM30DEM  improvement--   Elevation:44.97%    Slope:-17.33%   Curvature:-16.08%
---Fathom RMSE  Elevation: 3.88,  Slope: 2.22, Curvature: 0.0601
---FathomDEM  improvement--   Elevation:64.85%    Slope:13.92%   Curvature:7.53%


In [11]:
predRF=pred_val_MLModel(model='rf', modelPath=r"DataModels/RF_All_NewZeaLand.pkl")

---------Model: rf---------
-----Raw RMSE--- Elevation:3.73  
-----RMSE Improvement--- Elevation:66.16%  
-----Raw RMSE--- Elevation:3.73   Slope:2.17   Curvature:0.0574 
-----RMSE Improvement--- Elevation:66.16%             Slope:15.76%            Curvature:11.70% 


In [ ]:
predRF=pred_val_MLModel(model='extra_tree', modelPath=r"DataModels/ETF_NewZeaLand.pkl")

---------Model: extra_tree---------
-----Raw RMSE--- Elevation:3.82  
-----RMSE Improvement--- Elevation:65.42%  
-----Raw RMSE--- Elevation:3.82   Slope:2.19   Curvature:0.0580 
-----RMSE Improvement--- Elevation:65.42%             Slope:15.00%            Curvature:10.81% 


In [ ]:
predXGB=pred_val_MLModel(model="xgboost", modelPath=r"DataModels/XGBoost_NewZeaLand.pkl")

---------Model: xgboost---------
-----Raw RMSE--- Elevation:3.58  
-----RMSE Improvement--- Elevation:67.60%  
-----Raw RMSE--- Elevation:3.58   Slope:2.11   Curvature:0.0559 
-----RMSE Improvement--- Elevation:67.60%             Slope:18.28%            Curvature:13.91% 


In [ ]:
predCat=pred_val_MLModel(model="catboost", modelPath=r"Models/FinalModels/Catboost_NewZeaLand.pkl" )

---------Model: catboost---------
-----Raw RMSE--- Elevation:3.72  
-----RMSE Improvement--- Elevation:66.34%  
-----Raw RMSE--- Elevation:3.72   Slope:2.16   Curvature:0.0572 
-----RMSE Improvement--- Elevation:66.34%             Slope:16.26%            Curvature:12.05% 


In [ ]:
predCat=pred_val_MLModel(model="lgbm", modelPath=r"DataModels/LightGBM_NewZeaLand.pkl")

---------Model: lgbm---------
-----Raw RMSE--- Elevation:3.53  
-----RMSE Improvement--- Elevation:68.03%  
-----Raw RMSE--- Elevation:3.53   Slope:2.10   Curvature:0.0556 
-----RMSE Improvement--- Elevation:68.03%             Slope:18.54%            Curvature:14.45% 


In [ ]:
predANN=predANN(modelPath=r"DataModels/ANN_NewZeaLand.pth") 

ANN...........


-----Raw RMSE--- Elevation:3.81  
-----Raw RMSE--- Elevation:3.81   Slope:2.21   Curvature:0.0585 
-----RMSE Improvement--- Elevation:65.44%             Slope:14.33%            Curvature:10.02% 


In [ ]:
predCat=pred_val_MLModel(model="sgd", modelPath=r"DataModels/SGD_NewZeaLand.pkl")

---------Model: sgd---------
-----Raw RMSE--- Elevation:4.97  
-----RMSE Improvement--- Elevation:55.01%  
-----Raw RMSE--- Elevation:4.97   Slope:2.42   Curvature:0.0627 
-----RMSE Improvement--- Elevation:55.01%             Slope:6.09%            Curvature:3.46% 


In [ ]:

testPaths =  [testPath_2] 
otherPaths= [otherPath_2]

mean_std_path=r"DataModels/mean_std.npz"
test_data_2d,test_labels_1d=getTrainingData_2d(testPaths, mean_std_path)
pred_Fathom,pred_FABDEM,pred_GEDTM30=load_other_data(otherPaths)




---RawDEM RMSE  Elevation: 11.99,  Slope: 3.18, Curvature: 0.0473
---FABDEM RMSE  Elevation: 5.47,  Slope: 2.99, Curvature: 0.0452
---FABDEM  improvement--   Elevation:54.39%    Slope:5.94%   Curvature:4.43%
---GEDTM30 RMSE  Elevation: 5.15,  Slope: 3.00, Curvature: 0.0464
---GEDTM30DEM  improvement--   Elevation:57.02%    Slope:5.54%   Curvature:2.01%
---Fathom RMSE  Elevation: 3.13,  Slope: 1.92, Curvature: 0.0322
---FathomDEM  improvement--   Elevation:73.93%    Slope:39.52%   Curvature:31.84%


In [ ]:
predCat=pred_val_MLModel(model="rf", modelPath=r"DataModels/RF_America.pkl" )

---------Model: rf---------
-----Raw RMSE--- Elevation:3.78  
-----RMSE Improvement--- Elevation:68.43%  
-----Raw RMSE--- Elevation:3.78   Slope:2.28   Curvature:0.0366 
-----RMSE Improvement--- Elevation:68.43%             Slope:28.29%            Curvature:22.75% 


In [ ]:
predCat=pred_val_MLModel(model="extra_tree", modelPath=r"DataModels/ETF_America.pkl", IsValidation=True, time=150 )

---------Model: extra_tree---------
-----Raw RMSE--- Elevation:3.84  
-----RMSE Improvement--- Elevation:68.01%  
-----Raw RMSE--- Elevation:3.84   Slope:2.33   Curvature:0.0372 
-----RMSE Improvement--- Elevation:68.01%             Slope:26.66%            Curvature:21.36% 


In [ ]:
predCat=pred_val_MLModel(model="lgbm", modelPath=r"DataModels/LightGBM_America.pkl")

---------Model: lgbm---------
-----Raw RMSE--- Elevation:3.54  
-----RMSE Improvement--- Elevation:70.47%  
-----Raw RMSE--- Elevation:3.54   Slope:2.25   Curvature:0.0357 
-----RMSE Improvement--- Elevation:70.47%             Slope:29.22%            Curvature:24.65% 


In [ ]:
predCat=pred_val_MLModel(model="xgboost", modelPath=r"DataModels/XGBoost_America.pkl" )

---------Model: xgboost---------
-----Raw RMSE--- Elevation:3.59  
-----RMSE Improvement--- Elevation:70.02%  
-----Raw RMSE--- Elevation:3.59   Slope:2.25   Curvature:0.0359 
-----RMSE Improvement--- Elevation:70.02%             Slope:29.22%            Curvature:24.11% 


In [ ]:
predCat=pred_val_MLModel(model="catboost", modelPath=r"DataModels/Catboost_America.pkl")

---------Model: catboost---------
-----Raw RMSE--- Elevation:3.78  
-----RMSE Improvement--- Elevation:68.47%  
-----Raw RMSE--- Elevation:3.78   Slope:2.29   Curvature:0.0368 
-----RMSE Improvement--- Elevation:68.47%             Slope:27.89%            Curvature:22.23% 


In [ ]:
predANN=predANN(modelPath=r"DataModels/ANN_America.pth") 

ANN...........
-----Raw RMSE--- Elevation:3.89  
-----Raw RMSE--- Elevation:3.89   Slope:2.35   Curvature:0.0376 
-----RMSE Improvement--- Elevation:67.52%             Slope:26.18%            Curvature:20.58% 


In [ ]:
predCat=pred_val_MLModel(model="sgd", modelPath=r"DataModels/SGD_America.pkl")

---------Model: sgd---------
-----Raw RMSE--- Elevation:4.85  
-----RMSE Improvement--- Elevation:59.53%  
-----Raw RMSE--- Elevation:4.85   Slope:2.60   Curvature:0.0402 
-----RMSE Improvement--- Elevation:59.53%             Slope:18.24%            Curvature:15.10% 


In [ ]:

testPaths =  [testPath_3] 
otherPaths= [otherPath_3]

mean_std_path=r"DataModels/mean_std.npz"
test_data_2d,test_labels_1d=getTrainingData_2d(testPaths, mean_std_path)
pred_Fathom,pred_FABDEM,pred_GEDTM30=load_other_data(otherPaths)



predRF=pred_val_MLModel(model='rf', modelPath=r"DataModels/RF_All_European.pkl")
print()
predRF=pred_val_MLModel(model='extra_tree', modelPath=r"DataModels/ETF_European.pkl")
print()
predXGB=pred_val_MLModel(model="xgboost", modelPath=r"DataModels/XGBoost_European.pkl")
print()
predCat=pred_val_MLModel(model="lgbm", modelPath=r"DataModels/LightGBM_European.pkl" )
print()
predCat=pred_val_MLModel(model="catboost", modelPath=r"DataModels//Catboost_European.pkl")
print()
predANN=predANN(modelPath=r"DataModels/ANN_European.pth") 
print()
predCat=pred_val_MLModel(model="sgd", modelPath=r"DataModels/SGD_European.pkl", )


---RawDEM RMSE  Elevation: 8.28,  Slope: 3.09, Curvature: 0.0787
---FABDEM RMSE  Elevation: 5.10,  Slope: 2.78, Curvature: 0.0786
---FABDEM  improvement--   Elevation:38.38%    Slope:10.05%   Curvature:0.19%
---GEDTM30 RMSE  Elevation: 4.92,  Slope: 2.78, Curvature: 0.0776
---GEDTM30DEM  improvement--   Elevation:40.56%    Slope:10.00%   Curvature:1.46%
---Fathom RMSE  Elevation: 3.77,  Slope: 1.99, Curvature: 0.0651
---FathomDEM  improvement--   Elevation:54.47%    Slope:35.66%   Curvature:17.34%
---------Model: rf---------
-----Raw RMSE--- Elevation:3.98  
-----RMSE Improvement--- Elevation:51.97%  
-----Raw RMSE--- Elevation:3.98   Slope:2.21   Curvature:0.0687 
-----RMSE Improvement--- Elevation:51.97%             Slope:28.36%            Curvature:12.71% 

---------Model: extra_tree---------
-----Raw RMSE--- Elevation:4.02  
-----RMSE Improvement--- Elevation:51.41%  
-----Raw RMSE--- Elevation:4.02   Slope:2.26   Curvature:0.0693 
-----RMSE Improvement--- Elevation:51.41%         